# Black Spot Syndrome (BSS) spot detection with YOLO — demo

This notebook walks through the first batch of expert annotations (24 solitary *Acanthurus tractus* photos, spots boxed by Rylee in [MakeSense](https://www.makesense.ai/) and exported in **YOLO format**) and shows how they plug straight into a YOLO object-detection model.

**What it covers**

1. Inspect and validate the YOLO label files
2. Fetch the matching photos from iNaturalist (the label filenames are iNaturalist *observation IDs*)
3. Draw the annotations on the images to sanity-check them
4. Train a small YOLO detector on the 24 images
5. Turn detections into **spot counts** and score them against the count targets in the requirements doc (MAE ≤ 2, RMSE ≤ 3, exact ≥ 80%, ±1 ≥ 90%)
6. Answer the two questions: *is this format/approach reasonable?* and *should we resize images before annotating?*

> ⚠️ 24 images is far too few to train a useful model — step 4 exists to prove the pipeline works end to end, not to measure real performance.

In [ ]:
# %pip install ultralytics opencv-python pandas matplotlib requests pyyaml
import json, math, random, shutil, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yaml

ROOT = Path.cwd()
LABEL_DIR = ROOT / "labels_bss-surgeon-1_25_2026-09-22-11-58-55"
IMG_DIR = ROOT / "images"            # downloaded iNaturalist photos
DATASET = ROOT / "bss_yolo_dataset"  # YOLO train/val layout
CLASS_NAMES = ["bss_spot"]           # MakeSense YOLO export has no names file; class 0 = spot

IMG_DIR.mkdir(exist_ok=True)
random.seed(0)
print(len(list(LABEL_DIR.glob("*.txt"))), "label files")

## 1. The YOLO label format

Each image gets one `.txt` file with the **same base name** as the image. Each line is one box:

```
<class_id> <x_center> <y_center> <width> <height>
```

All four coordinates are **normalised to 0–1** relative to the image width/height. That detail matters later: the labels don't depend on the image's pixel size.

In [ ]:
print((LABEL_DIR / "5483838.txt").read_text())

In [ ]:
rows, problems = [], []
for f in sorted(LABEL_DIR.glob("*.txt")):
    for i, line in enumerate(f.read_text().splitlines()):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            problems.append((f.name, i, "expected 5 fields"))
            continue
        c, x, y, w, h = int(parts[0]), *map(float, parts[1:])
        if c != 0:
            problems.append((f.name, i, f"unexpected class {c}"))
        if not all(0 <= v <= 1 for v in (x, y, w, h)):
            problems.append((f.name, i, "coordinate outside 0-1"))
        if w <= 0 or h <= 0:
            problems.append((f.name, i, "zero-size box"))
        rows.append(dict(obs_id=f.stem, cls=c, x=x, y=y, w=w, h=h))

boxes = pd.DataFrame(rows)
print("Total boxes:", len(boxes), "| problems:", problems or "none")
boxes.describe().round(4)

In [ ]:
counts = boxes.groupby("obs_id").size().rename("gt_spots")
counts = counts.reindex([f.stem for f in LABEL_DIR.glob("*.txt")], fill_value=0).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
counts.plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set(title="Spots annotated per image", xlabel="iNaturalist observation", ylabel="spots")
axes[1].scatter(boxes.w, boxes.h, s=12, alpha=.6)
axes[1].set(title="Box size (fraction of image)", xlabel="width", ylabel="height")
plt.tight_layout(); plt.show()
print(counts.describe().round(1))

Observations so far:

- **Two boxes have zero width or height** (in `13219574.txt` and `3137185.txt`). These are probably accidental clicks in MakeSense. YOLO drops them during training, but they're worth deleting at the source.
- Every file has **at least one box** — this batch contains only *infected* fish. The model also needs uninfected fish (empty label files) and the NA class later on.
- Spot counts are very skewed: 9 images have one spot, three have about 40.
- Spots are **tiny**: the median box covers about 1.6% × 2% of the image. On a 640 px input, which is YOLO's default, that's about 10 px per spot. That's the main technical risk, and it's the reason for the resizing advice below.

## 2. Fetch the images from iNaturalist

The label filenames (`5483838.txt`, ...) are iNaturalist observation IDs. We use the public API to find each observation's photo and download it. This uses the **first** photo of each observation, so for the few observations with several photos, check that it's the one Rylee annotated. The overlays in step 3 make a mismatch obvious.

If you already have the original files, drop them into `images/` named `<obs_id>.jpg` and this cell will skip them.

In [ ]:
def fetch_observation(obs_id):
    r = requests.get(f"https://api.inaturalist.org/v1/observations/{obs_id}", timeout=30)
    r.raise_for_status()
    return r.json()["results"][0]

meta = []
for obs_id in counts.index:
    out = IMG_DIR / f"{obs_id}.jpg"
    o = fetch_observation(obs_id)
    p = o["photos"][0]
    meta.append(dict(obs_id=obs_id, taxon=o["taxon"]["name"], n_photos=len(o["photos"]),
                     place=o.get("place_guess"), license=p.get("license_code"),
                     orig_w=p["original_dimensions"]["width"], orig_h=p["original_dimensions"]["height"]))
    if not out.exists():
        for size in ("original", "large"):  # 'original' isn't always public; 'large' is ≤1024 px
            url = p["url"].replace("square", size)
            resp = requests.get(url, timeout=60)
            if resp.ok:
                out.write_bytes(resp.content)
                break
    time.sleep(1)  # be polite to the iNaturalist API

meta = pd.DataFrame(meta).set_index("obs_id")
meta["downloaded_wh"] = [cv2.imread(str(IMG_DIR / f"{i}.jpg")).shape[1::-1] for i in meta.index]
meta.join(counts)

The photos range from about 750 px to 2048 px wide and come in several aspect ratios. That mix is normal for community-science data, and section 6 explains why it's fine.

## 3. Visualise the annotations

In [ ]:
def load_boxes(obs_id):
    f = LABEL_DIR / f"{obs_id}.txt"
    return np.loadtxt(f, ndmin=2) if f.read_text().strip() else np.zeros((0, 5))

def draw(img, yolo_boxes, color=(0, 255, 0), thickness=2):
    H, W = img.shape[:2]
    for _, x, y, w, h in yolo_boxes[:, :5]:
        x1, y1 = int((x - w / 2) * W), int((y - h / 2) * H)
        x2, y2 = int((x + w / 2) * W), int((y + h / 2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    return img

ids = list(counts.index)
fig, axes = plt.subplots(6, 4, figsize=(18, 22))
for ax, obs_id in zip(axes.flat, ids):
    img = cv2.cvtColor(cv2.imread(str(IMG_DIR / f"{obs_id}.jpg")), cv2.COLOR_BGR2RGB)
    ax.imshow(draw(img, load_boxes(obs_id), color=(255, 255, 0), thickness=max(2, img.shape[1] // 400)))
    ax.set_title(f"{obs_id}  —  {counts[obs_id]} spots"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Zoom in on the busiest image to check how tight the boxes are
obs_id = counts.idxmax()
img = cv2.cvtColor(cv2.imread(str(IMG_DIR / f"{obs_id}.jpg")), cv2.COLOR_BGR2RGB)
b = load_boxes(obs_id)
H, W = img.shape[:2]
x1, x2 = int((b[:, 1].min() - .05) * W), int((b[:, 1].max() + .05) * W)
y1, y2 = int((b[:, 2].min() - .05) * H), int((b[:, 2].max() + .05) * H)
plt.figure(figsize=(14, 8))
plt.imshow(draw(img.copy(), b, color=(255, 255, 0))[max(y1, 0):y2, max(x1, 0):x2])
plt.title(f"{obs_id}: {len(b)} annotated spots (zoomed)"); plt.axis("off"); plt.show()

## 4. Build a YOLO dataset and train

Ultralytics expects this layout:

```
bss_yolo_dataset/
  images/{train,val}/<id>.jpg
  labels/{train,val}/<id>.txt
  data.yaml
```

With only 24 images we hold out 6 for validation, stratified so that both splits include low-count and high-count fish.

In [ ]:
order = list(counts.sort_values().index)
val_ids = order[1::4]                       # every 4th image along the count ranking -> 6 val images
train_ids = [i for i in order if i not in val_ids]

if DATASET.exists():
    shutil.rmtree(DATASET)
for split, split_ids in (("train", train_ids), ("val", val_ids)):
    (DATASET / "images" / split).mkdir(parents=True)
    (DATASET / "labels" / split).mkdir(parents=True)
    for i in split_ids:
        shutil.copy(IMG_DIR / f"{i}.jpg", DATASET / "images" / split / f"{i}.jpg")
        shutil.copy(LABEL_DIR / f"{i}.txt", DATASET / "labels" / split / f"{i}.txt")

data_yaml = DATASET / "data.yaml"
data_yaml.write_text(yaml.safe_dump({"path": str(DATASET), "train": "images/train",
                                     "val": "images/val", "names": dict(enumerate(CLASS_NAMES))}))
print(data_yaml.read_text())
print("train:", len(train_ids), "val:", len(val_ids), "| val spot counts:", counts[val_ids].tolist())

Training settings, chosen for **tiny objects**:

- `imgsz=1280`: at 640 the spots shrink to a few pixels and become hard to detect. Larger inputs keep them visible.
- `yolo11n.pt` (COCO-pretrained, smallest): transfer learning matters when you only have 18 training images.
- Augmentation: flips and colour jitter help, but `mosaic` can shrink tiny objects further, so we switch it off for the last epochs.

`EPOCHS` is kept small so the demo runs on a laptop CPU. Raise it, and use a GPU, for real experiments.

In [ ]:
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"   # 'mps' crashed mid-training in our tests (Ultralytics 8.3)
EPOCHS, IMGSZ = 40, 1280

model = YOLO("yolo11n.pt")
results = model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=4, device=DEVICE,
                      fliplr=0.5, flipud=0.0, close_mosaic=10, patience=0,
                      project=str(ROOT / "runs"), name="bss_demo", verbose=False, plots=True)  # each run gets its own folder (bss_demo2, ...)

In [ ]:
run_dir = Path(results.save_dir)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, name in zip(axes, ["results.png", "val_batch0_pred.jpg"]):
    p = run_dir / name
    if p.exists():
        ax.imshow(plt.imread(p)); ax.set_title(name)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. From detections to spot counts

For the BSS use case the detector is a means to an end: **spot count = number of boxes above a confidence threshold.** We pick the threshold on the training images (it's the knob that trades missed spots against false spots), then report the count metrics from the requirements doc on the held-out images.

In [ ]:
best = YOLO(run_dir / "weights" / "best.pt")

def predict_counts(ids, conf=0.05):
    out = {}
    for i in ids:
        r = best.predict(IMG_DIR / f"{i}.jpg", imgsz=IMGSZ, conf=conf, device=DEVICE, verbose=False)[0]
        out[i] = r.boxes.conf.cpu().numpy()
    return out

def count_metrics(gt, pred):
    e = np.asarray(pred) - np.asarray(gt)
    ss_tot = ((np.asarray(gt) - np.mean(gt)) ** 2).sum()
    return dict(MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean()),
                R2=1 - (e ** 2).sum() / ss_tot if ss_tot else np.nan,
                exact=(e == 0).mean(), within_1=(np.abs(e) <= 1).mean())

train_conf = predict_counts(train_ids)
sweep = pd.DataFrame({t: count_metrics(counts[train_ids], [(c >= t).sum() for c in train_conf.values()])
                      for t in np.arange(0.05, 0.8, 0.05).round(2)}).T
CONF = sweep.MAE.idxmin()
print("chosen confidence threshold:", CONF)
sweep.round(2)

In [ ]:
val_conf = predict_counts(val_ids)
report = pd.DataFrame({"gt_spots": counts[val_ids],
                       "pred_spots": [(val_conf[i] >= CONF).sum() for i in val_ids]})
report["error"] = report.pred_spots - report.gt_spots
display(report)

targets = dict(MAE=("≤", 2), RMSE=("≤", 3), R2=("≥", .80), exact=("≥", .80), within_1=("≥", .90))
m = count_metrics(report.gt_spots, report.pred_spots)
pd.DataFrame([{"metric": k, "value": round(v, 2), "target": f"{targets[k][0]} {targets[k][1]}",
               "meets": (v <= targets[k][1]) if targets[k][0] == "≤" else (v >= targets[k][1])}
              for k, v in m.items()])

In [ ]:
# Ground truth (yellow) vs predictions (magenta) on the validation images
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, i in zip(axes.flat, val_ids):
    img = cv2.cvtColor(cv2.imread(str(IMG_DIR / f"{i}.jpg")), cv2.COLOR_BGR2RGB)
    t = max(2, img.shape[1] // 400)
    r = best.predict(IMG_DIR / f"{i}.jpg", imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]
    pred = np.c_[np.zeros(len(r.boxes)), r.boxes.xywhn.cpu().numpy()]
    img = draw(img, load_boxes(i), color=(255, 255, 0), thickness=t)
    img = draw(img, pred, color=(255, 0, 255), thickness=t)
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{i}: GT {counts[i]}  |  pred {len(pred)}")
plt.tight_layout(); plt.show()

## 6. Takeaways for the team

### Is the MakeSense + YOLO-format + rectangle-per-spot approach reasonable?

**Yes. It's a good starting point.**

- YOLO txt is the native format for Ultralytics and most modern detectors, and it converts easily to COCO and others. The notebook used Rylee's export **as-is**: the only thing we added was `data.yaml` with the class name, because MakeSense exports don't include one.
- One box per spot gives you **detection *and* counting in one model**: count = number of boxes. That matches the "spot count" output in the requirements.
- The files passed validation: correct field count, a single class, every coordinate inside 0–1, no zero-size boxes.

Things to fix as annotation scales up:

1. **Keep the image ↔ label link explicit.** Labels are named by iNaturalist observation ID, and some observations have more than one photo. Record the exact photo ID (or keep the image file itself) next to each label file.
2. **Annotate negatives and NA cases.** This batch has only infected fish. The model also needs uninfected fish (an *empty* `.txt` file = "no spots") and NA fish so it learns not to over-call. Natural markings, scars and other lesions are especially valuable as hard negatives.
3. **Write a short spot-boxing guideline** (e.g. "box each distinct black spot; merge or split touching spots by rule X; ignore spots smaller than Y") and have a second annotator double-label about 10% of images. Inter-annotator agreement is the human baseline that the "at least as well as a human" target needs.
4. **Plan for the fish/NA stage.** The requirements describe fish detection or quality gating (Y/N/NA) *before* spot counting. A two-stage pipeline (detect each fish → crop → count spots on the crop) also handles multi-fish images and makes the tiny spots bigger relative to the model input.

### Should we resize all images to the same size before annotating?

**No. Annotate the originals at full resolution.**

- YOLO labels are **normalised (0–1)**, so they stay valid at any resolution. The training pipeline resizes and letterboxes every image to `imgsz` automatically. Mixed sizes (745–2048 px in this batch) are fine.
- Shrinking images before annotating permanently loses detail, and **spots are tiny** (about 1–3% of image width). Annotators need full resolution to see them, and we may later want to train at higher resolution or on **fish crops/tiles**, which only works if the originals are kept.
- Resizing *does* matter at **training** time. Use a large `imgsz` (1024–1280+), crop to the fish, or use tiled inference (e.g. SAHI) so each spot stays at least ~10–15 px on the model's input.
- One caveat: if you edit images at all (crop, rotate, resize) **after** annotation, regenerate or transform the labels. Normalised coordinates survive a uniform resize but not a crop.